In [3]:
from pathlib import Path
import sys

repo_root = Path.cwd().parent  # if the notebook's working directory is notebooks/
sys.path.insert(0, str(repo_root))

In [4]:
# define categories for classifier

relevant_categories = [
    "Cap On Liability", "Termination For Convenience", "Governing Law",
    "Anti-Assignment", "Change Of Control", "Renewal Term",
    "Exclusivity", "Non-Compete", "Notice Period To Terminate Renewal", "Liquidated Damages"
]

relevant_categories = [category.casefold() for category in relevant_categories]

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import MultiLabelBinarizer
import import_ipynb
from src.chunking import tokenizer, json_to_df_new
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression


In [6]:
def prepare_data(path, tokenizer, mlb, relevant_categories):
    chunks = json_to_df_new(path, tokenizer)
    chunks = chunks[
        chunks['categories'].apply(lambda labels: any(label in relevant_categories for label in labels))
    ].copy()
    chunks['categories'] = chunks['categories'].apply(
        lambda labels: [label for label in labels if label in relevant_categories]
    )
    X = chunks['text'].fillna('').astype(str)
    y = mlb.fit_transform(chunks['categories'])
    return X, y

In [7]:
# load, chunk, vectorize training data
project_root = Path.cwd().parent
mlb = MultiLabelBinarizer(classes=relevant_categories)
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=2)
X_train, y_train = prepare_data(project_root / "data/train_separate_questions.json", 
                                tokenizer, mlb, relevant_categories)
X_train_vec = vectorizer.fit_transform(X_train)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (7556 > 512). Running this sequence through the model will result in indexing errors


In [8]:
# load, chunk, vectorize test data
X_test, y_test = prepare_data(project_root / 'data/test.json', tokenizer, 
                              mlb, relevant_categories) 
X_test_vec = vectorizer.transform(X_test)

In [9]:
# Logistic Regression baseline
logreg = LogisticRegression(max_iter=2000, class_weight='balanced', solver='liblinear')
logreg_classifier = OneVsRestClassifier(logreg)
logreg_classifier.fit(X_train_vec, y_train)
logreg_predictions = logreg_classifier.predict(X_test_vec)

print('Logistic Regression multilabel results:')
print(classification_report(y_test, logreg_predictions, target_names=mlb.classes_, zero_division=0))

Logistic Regression multilabel results:
                                    precision    recall  f1-score   support

                  cap on liability       0.89      0.78      0.83       112
       termination for convenience       0.49      0.67      0.56        52
                     governing law       0.81      0.93      0.87       116
                   anti-assignment       0.78      0.79      0.78       126
                 change of control       0.66      0.48      0.56        60
                      renewal term       0.67      0.88      0.76        40
                       exclusivity       0.62      0.79      0.69        81
                       non-compete       0.67      0.74      0.70        58
notice period to terminate renewal       0.45      0.79      0.58        24
                liquidated damages       0.91      0.42      0.57        24

                         micro avg       0.71      0.76      0.73       693
                         macro avg       0.69 

In [10]:
#LinearSVC model
svc = LinearSVC(class_weight='balanced')
svc_classifier = OneVsRestClassifier(svc)
svc_classifier.fit(X_train_vec, y_train)
svc_predictions = svc_classifier.predict(X_test_vec)

print('LinearSVC multilabel results:')
print(classification_report(y_test, svc_predictions, target_names=mlb.classes_, zero_division=0))


LinearSVC multilabel results:
                                    precision    recall  f1-score   support

                  cap on liability       0.91      0.77      0.83       112
       termination for convenience       0.62      0.62      0.62        52
                     governing law       0.96      0.92      0.94       116
                   anti-assignment       0.82      0.71      0.76       126
                 change of control       0.96      0.37      0.53        60
                      renewal term       0.83      0.75      0.79        40
                       exclusivity       0.70      0.67      0.68        81
                       non-compete       0.87      0.59      0.70        58
notice period to terminate renewal       0.64      0.58      0.61        24
                liquidated damages       1.00      0.33      0.50        24

                         micro avg       0.83      0.69      0.75       693
                         macro avg       0.83      0.63 

The LinearSVC model appears to perform better than the Logistic Regression model and should be used as the baseline.